# Normalize image

In [3]:
def percentile_normalize(img, lower=1, upper=99):
    img = img.astype(np.float32)

    p_low, p_high = np.percentile(img, (lower, upper))
    img = np.clip(img, p_low, p_high)

    img = (img - p_low) / (p_high - p_low + 1e-8)
    img = img * 2 - 1
    return img

In [25]:
t1_path = "atlas/T1/sub-*.nii.gz"

t1_path = sorted(glob.glob(t1_path))
for i, t1 in enumerate(tqdm(t1_path)):
    # if i > 0:
    #     break
    sub_id = t1.split("/")[-1].split(".")[0]

    img = nib.load(t1)
    img_affine = img.affine
    
    img = img.get_fdata()
    img = percentile_normalize(img)

    save_path = f"dataset/atlas/image/{sub_id}.nii.gz"
    nib.save(nib.Nifti1Image(img, affine=img_affine), save_path)

100%|██████████| 655/655 [04:23<00:00,  2.49it/s]


In [28]:
# # plt.imshow(img[:, :, img.shape[2]//2], cmap="gray")
# # plt.axis("off")

# plt.hist(img.flatten(), bins=100)
# plt.ylim(0, 0.2e6)
# # plt.xlim(-0.5,1)
# plt.show()

# Resize image and mask

In [9]:
import torchio as tio
import glob
from tqdm import tqdm
import nibabel as nib
import numpy as np

In [3]:
def resize(img, mask, target_size=(128,128,128)):
    subject = tio.Subject(
        image=tio.ScalarImage(tensor=img[np.newaxis,...]),
        mask=tio.LabelMap(tensor=mask[np.newaxis,...])
    )

    transform = tio.Resize(
        target_size,
        image_interpolation='linear',
        label_interpolation='nearest'
    )

    out = transform(subject)

    img = out['image'].data.numpy().squeeze(0)
    mask = out['mask'].data.numpy().squeeze(0)

    return img, mask

In [14]:
img_path = "dataset/atlas/image/sub-*.nii.gz"
# mask_path = "dataset/atlas/mask/sub-*.nii.gz"

img_path = sorted(glob.glob(img_path))
# mask_path = sorted(glob.glob(mask_path))

for i, img in enumerate(tqdm(img_path)):
    # if i > 0:
    #     break
    sub_id = img.split("/")[-1].split(".")[0]
    mask = f"dataset/atlas/mask/{sub_id}.nii.gz"

    img = nib.load(img)
    mask = nib.load(mask)
    img_affine = img.affine
    mask_affine = mask.affine
    assert (img_affine == mask_affine).all()

    img = img.get_fdata()
    mask = mask.get_fdata()

    img_resized, mask_resized = resize(img, mask)

    save_img_path = f"dataset/atlas/image/{sub_id}.nii.gz"
    save_mask_path = f"dataset/atlas/mask/{sub_id}.nii.gz"

    nib.save(nib.Nifti1Image(img_resized, affine=img_affine), save_img_path)
    nib.save(nib.Nifti1Image(mask_resized, affine=mask_affine), save_mask_path)


100%|██████████| 655/655 [04:29<00:00,  2.43it/s]
